# Thống kê Gloss trong Dataset VSL400

Notebook này thống kê dataset **VSL400** (Vietnamese Word-Level Sign Language Recognition):
- Số lượng video theo từng gloss
- Số lượng người sign (signer) khác nhau theo từng gloss

Dataset trên Kaggle có cấu trúc:
```
vsl400/
├── Part_1/
│   └── split_1/
│       ├── front_view/          (video .mp4)
│       ├── left_view/
│       ├── right_view/
│       ├── front_view.json      (metadata)
│       ├── left_view.json
│       └── right_view.json
├── Part_2/
│   └── split_2/
...
├── Part_7/
│   └── split_7/
```

Mỗi entry trong file json có dạng:
```json
{
  "video_id": "000000",
  "signer_id": "001",
  "fps": 25,
  "resolution": 1080,
  "gloss": "Anh",
  "num_frames": 65,
  "length_seconds": 2.6
}
```

Vì cả 3 view (front/left/right) của cùng một `video_id` mô tả **cùng một lần thực hiện gesture** (cùng signer,
cùng gloss), chỉ cần dùng metadata của **1 view** (chọn `front_view.json`) để đếm số video/signer theo gloss, tránh đếm trùng x3.

In [1]:
import json
import glob
from pathlib import Path
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 100)
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Tìm tất cả các split trong dataset

Đoạn code dưới dò tìm tự động tất cả các thư mục chứa `front_view.json` bên trong thư mục input của Kaggle,
nên vẫn hoạt động dù cấu trúc lồng nhau là `Part_x/split_x/...` hay `split_x/...` trực tiếp.

In [2]:
# Đường dẫn gốc của dataset trên Kaggle
DATASET_ROOT = Path("D:\Hoc Tap\VSL Recognition System\VSL-Recognition-System")

print("Dataset root:", DATASET_ROOT)

# Tìm mọi file front_view.json trong toàn bộ cây thư mục -> mỗi file tương ứng 1 split
front_json_paths = sorted(DATASET_ROOT.rglob("front_view.json"))

print(f"Tìm thấy {len(front_json_paths)} split (front_view.json):")
for p in front_json_paths:
    print(" -", p)

Dataset root: D:\Hoc Tap\VSL Recognition System\VSL-Recognition-System
Tìm thấy 7 split (front_view.json):
 - D:\Hoc Tap\VSL Recognition System\VSL-Recognition-System\data\raw\Part_1\split_1\front_view.json
 - D:\Hoc Tap\VSL Recognition System\VSL-Recognition-System\data\raw\Part_2\split_2\front_view.json
 - D:\Hoc Tap\VSL Recognition System\VSL-Recognition-System\data\raw\Part_3\split_3\front_view.json
 - D:\Hoc Tap\VSL Recognition System\VSL-Recognition-System\data\raw\Part_4\split_4\front_view.json
 - D:\Hoc Tap\VSL Recognition System\VSL-Recognition-System\data\raw\Part_5\split_5\front_view.json
 - D:\Hoc Tap\VSL Recognition System\VSL-Recognition-System\data\raw\Part_6\split_6\front_view.json
 - D:\Hoc Tap\VSL Recognition System\VSL-Recognition-System\data\raw\Part_7\split_7\front_view.json


## 2. Nạp metadata từ tất cả các split

Chỉ nạp `front_view.json` của mỗi split (đại diện cho cả 3 view) để tránh đếm trùng.

In [4]:
records = []

for split_json in front_json_paths:
    split_name = split_json.parent.name  # ví dụ: split_1
    with open(split_json, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        records.append({
            "split": split_name,
            "video_id": item["video_id"],
            "signer_id": item["signer_id"],
            "gloss": item["gloss"],
            "fps": item.get("fps"),
            "resolution": item.get("resolution"),
            "num_frames": item.get("num_frames"),
            "length_seconds": item.get("length_seconds"),
        })

df = pd.DataFrame(records)
print(f"Tổng số video (instances) nạp được: {len(df)}")
print(f"Số split: {df['split'].nunique()}")
print(f"Số gloss khác nhau: {df['gloss'].nunique()}")
print(f"Số signer khác nhau: {df['signer_id'].nunique()}")
df.head(400)

Tổng số video (instances) nạp được: 24753
Số split: 7
Số gloss khác nhau: 400
Số signer khác nhau: 26


,split,video_id,signer_id,gloss,fps,resolution,num_frames,length_seconds
0,split_1,000000,001,Anh,25.0,1080,65,2.60
1,split_1,000001,001,Anh,25.0,1080,56,2.24
2,split_1,000002,001,Chị,25.0,1080,62,2.48
3,split_1,000003,001,Chị,25.0,1080,76,3.04
4,split_1,000004,001,Em,25.0,1080,65,2.60
...,...,...,...,...,...,...,...,...
395,split_1,000395,001,Thư ký,25.0,1080,67,2.68
396,split_1,000396,001,Thư ký,25.0,1080,61,2.44
397,split_1,000397,001,Máy bay,25.0,1080,63,2.52
398,split_1,000398,001,Máy bay,25.0,1080,60,2.40


## 3. Kiểm tra sanity (video_id trùng lặp giữa các split)

`video_id` được kỳ vọng là duy nhất trên toàn bộ dataset (mỗi split chỉ chứa 1 khoảng ID riêng). Kiểm tra lại
để chắc chắn không có video nào bị đếm 2 lần.

In [5]:
dup_count = df["video_id"].duplicated().sum()
if dup_count > 0:
    print(f"Cảnh báo: có {dup_count} video_id bị trùng giữa các split — loại bỏ bản trùng trước khi thống kê.")
    df = df.drop_duplicates(subset="video_id", keep="first")
else:
    print("Không có video_id trùng lặp giữa các split.")

Không có video_id trùng lặp giữa các split.


## 4. Thống kê theo gloss: số video & số signer

In [6]:
gloss_stats = (
    df.groupby("gloss")
      .agg(
          num_videos=("video_id", "nunique"),
          num_signers=("signer_id", "nunique"),
      )
      .reset_index()
      .sort_values("num_videos", ascending=False)
      .reset_index(drop=True)
)

print(f"Tổng số gloss: {len(gloss_stats)}")
gloss_stats.head(20)

Tổng số gloss: 400


,gloss,num_videos,num_signers
0,Mùa hè,93,26
1,Ngày,89,25
2,Nắng,87,26
3,Ướt,86,26
4,Nhẹ,85,26
5,Em,84,26
6,Anh,84,26
7,Thơm,82,26
8,Chị,81,26
9,Dễ,81,26


In [7]:
summary = {
    "total_glosses": int(gloss_stats.shape[0]),
    "total_videos": int(df["video_id"].nunique()),
    "total_signers": int(df["signer_id"].nunique()),
    "avg_videos_per_gloss": float(gloss_stats["num_videos"].mean()),
    "median_videos_per_gloss": float(gloss_stats["num_videos"].median()),
    "min_videos_per_gloss": int(gloss_stats["num_videos"].min()),
    "max_videos_per_gloss": int(gloss_stats["num_videos"].max()),
    "avg_signers_per_gloss": float(gloss_stats["num_signers"].mean()),
    "min_signers_per_gloss": int(gloss_stats["num_signers"].min()),
    "max_signers_per_gloss": int(gloss_stats["num_signers"].max()),
}

for k, v in summary.items():
    print(f"{k}: {v}")

total_glosses: 400
total_videos: 24753
total_signers: 26
avg_videos_per_gloss: 61.8825
median_videos_per_gloss: 59.0
min_videos_per_gloss: 53
max_videos_per_gloss: 93
avg_signers_per_gloss: 25.9875
min_signers_per_gloss: 23
max_signers_per_gloss: 26


## 5. Xuất kết quả ra file JSON

In [ ]:
output = {
    "summary": summary,
    "gloss_stats": gloss_stats.to_dict(orient="records"),
}

output_path = Path("D:\Hoc Tap\VSL Recognition System\VSL-Recognition-System")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Đã ghi kết quả vào: {output_path}")

OSError: [Errno 22] Invalid argument: 'D:\\Hoc Tap\\VSL Recognition System\\VSL-Recognition-System\reports'

## 6. Trực quan hóa

- Top 20 gloss có nhiều video nhất
- Top 20 gloss có nhiều signer nhất
- Phân bố số video / số signer trên mỗi gloss

In [ ]:
top_n = 20
top_videos = gloss_stats.nlargest(top_n, "num_videos")

fig, ax = plt.subplots()
ax.bar(top_videos["gloss"], top_videos["num_videos"], color="#4C72B0")
ax.set_title(f"Top {top_n} gloss theo số lượng video")
ax.set_xlabel("Gloss")
ax.set_ylabel("Số video")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
top_signers = gloss_stats.nlargest(top_n, "num_signers")

fig, ax = plt.subplots()
ax.bar(top_signers["gloss"], top_signers["num_signers"], color="#DD8452")
ax.set_title(f"Top {top_n} gloss theo số lượng signer")
ax.set_xlabel("Gloss")
ax.set_ylabel("Số signer")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(gloss_stats["num_videos"], bins=30, color="#55A868")
axes[0].set_title("Phân bố số video của mỗi gloss")
axes[0].set_xlabel("Số video")
axes[0].set_ylabel("Số lượng gloss")

axes[1].hist(gloss_stats["num_signers"], bins=30, color="#C44E52")
axes[1].set_title("Phân bố số signer trên mỗi gloss")
axes[1].set_xlabel("Số signer")
axes[1].set_ylabel("Số lượng gloss")

plt.tight_layout()
plt.show()

## 7. Thống kê num_frames và length_seconds

fps và resolution đã biết là đồng nhất (25fps, 1080p) trên toàn dataset nên không cần thống kê lại. Ở đây chỉ
xem phân bố `num_frames` và `length_seconds` để biết độ dài video dao động thế nào (phục vụ việc quyết định
độ dài clip/padding khi merge với dataset khác).

In [ ]:
numeric_stats = df[["num_frames", "length_seconds"]].describe()
print(numeric_stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["num_frames"], bins=40, color="#4C72B0")
axes[0].set_title("Phân bố num_frames")
axes[0].set_xlabel("Số frame")
axes[0].set_ylabel("Số video")

axes[1].hist(df["length_seconds"], bins=40, color="#DD8452")
axes[1].set_title("Phân bố length_seconds")
axes[1].set_xlabel("Thời lượng (giây)")
axes[1].set_ylabel("Số video")

plt.tight_layout()
plt.show()

## 8. Kiểm tra tính toàn vẹn dữ liệu (integrity check)

Hai việc kiểm tra, đều chỉ đọc **tên file/metadata**, không mở nội dung video nên vẫn nhanh dù dataset 72GB:

1. **Đồng bộ `video_id` giữa 3 view** trong cùng 1 split — front/left/right phải có cùng tập `video_id`.
2. **Khớp giữa JSON và file thực tế** — mọi `video_id` trong JSON phải có file `.mp4` tương ứng trong thư mục
   view, và không có file `.mp4` "mồ côi" (tồn tại nhưng không có trong JSON).

In [ ]:
# 8.1 Kiểm tra đồng bộ video_id giữa front / left / right trong từng split
mismatch_report = []

for split_json in front_json_paths:
    split_dir = split_json.parent
    front_path = split_dir / "front_view.json"
    left_path = split_dir / "left_view.json"
    right_path = split_dir / "right_view.json"

    with open(front_path, encoding="utf-8") as f:
        front_ids = {item["video_id"] for item in json.load(f)}
    with open(left_path, encoding="utf-8") as f:
        left_ids = {item["video_id"] for item in json.load(f)}
    with open(right_path, encoding="utf-8") as f:
        right_ids = {item["video_id"] for item in json.load(f)}

    missing_in_left = front_ids - left_ids
    missing_in_right = front_ids - right_ids
    extra_in_left = left_ids - front_ids
    extra_in_right = right_ids - front_ids

    if missing_in_left or missing_in_right or extra_in_left or extra_in_right:
        mismatch_report.append({
            "split": split_dir.name,
            "missing_in_left": len(missing_in_left),
            "missing_in_right": len(missing_in_right),
            "extra_in_left": len(extra_in_left),
            "extra_in_right": len(extra_in_right),
        })

if mismatch_report:
    print("Phát hiện không đồng bộ video_id giữa các view:")
    for r in mismatch_report:
        print(r)
else:
    print("video_id đồng bộ giữa front/left/right ở tất cả các split.")

In [ ]:
# 8.2 Kiểm tra khớp giữa metadata JSON và file .mp4 thực tế trên đĩa
file_check_report = []

for split_json in front_json_paths:
    split_dir = split_json.parent
    for view in ["front_view", "left_view", "right_view"]:
        json_path = split_dir / f"{view}.json"
        video_dir = split_dir / view

        with open(json_path, encoding="utf-8") as f:
            expected_ids = {item["video_id"] for item in json.load(f)}

        # .glob chỉ liệt kê tên file, không đọc nội dung video -> vẫn nhanh với dataset lớn
        actual_files = {p.stem for p in video_dir.glob("*.mp4")} if video_dir.exists() else set()

        missing_files = expected_ids - actual_files   # có trong JSON nhưng thiếu file
        orphan_files = actual_files - expected_ids     # có file nhưng không có trong JSON

        if missing_files or orphan_files:
            file_check_report.append({
                "split": split_dir.name,
                "view": view,
                "missing_files": len(missing_files),
                "orphan_files": len(orphan_files),
            })

if file_check_report:
    print("Phát hiện thiếu/lệch file video so với metadata:")
    for r in file_check_report:
        print(r)
else:
    print("Tất cả video_id trong metadata đều có file .mp4 tương ứng, không có file thừa.")